In [ ]:
# Installing required packages for Llama-3.1-8B training
!pip install -q --upgrade transformers datasets accelerate bitsandbytes peft trl sentencepiece 2>/dev/null
!pip install -q einops 2>/dev/null  # Required for Llama architectures

print("All packages installed!")

In [ ]:
# Configuration for Llama-3.1-8B on Kaggle
MODEL_ID = "meta-llama/Llama-3.1-8B"  # Llama-3.1-8B base model
DATASET_PATH = "/kaggle/input/datasets"  # Path to your dataset (adjust if needed)
OUTPUT_DIR = "/kaggle/working/llama31_logic_classifier"

# Output categories in your dataset:
OUTPUT_CATEGORIES = [
    "SIMPLE INSTRUCTION",
    "INSTRUCTION WITH SEQUENCE",
    "PARALLEL INSTRUCTION",
    "INSTRUCTION WITH PURPOSE",
    "INSTRUCTION WITH REASON",
    "EXCLUSIVE INSTRUCTION (OBJECTS)",
    "EXCLUSIVE INSTRUCTION (ACTIONS)"
]

MAX_SEQ_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

# Checkpoint/version retention settings
SAVE_EVERY_STEPS = 50
SAVE_EVERY_MINUTES = 30
KEEP_CHECKPOINTS = 2  # Keep latest + best-loss checkpoint when possible

# Kaggle secrets for Hugging Face token (supports multiple secret names)
from kaggle_secrets import UserSecretsClient
HF_TOKEN = None
try:
    user_secrets = UserSecretsClient()
    for secret_name in ["HF_TOKEN", "LLAMA_API", "Mistral_API"]:
        try:
            HF_TOKEN = user_secrets.get_secret(secret_name)
            if HF_TOKEN:
                print(f"Hugging Face token loaded from Kaggle secret: {secret_name}")
                break
        except Exception:
            continue
except Exception as e:
    print(f"Could not initialize Kaggle secrets client: {e}")

if HF_TOKEN is None:
    print("No Hugging Face token found. Set HF_TOKEN or LLAMA_API in Kaggle Secrets.")

print("Configuration loaded for Llama-3.1-8B!")
print(f"Model: {MODEL_ID}")
print(f"Dataset: {DATASET_PATH}")
print(f"Output categories: {len(OUTPUT_CATEGORIES)}")

In [ ]:
# Importing libraries
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {total_mem_gb:.1f} GB")
else:
    print("GPU: CPU (CUDA not available)")

print(f"PyTorch: {torch.__version__}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Loading dataset from Kaggle
print(f"Loading dataset from: {DATASET_PATH}")

if os.path.exists(DATASET_PATH):
    print("Dataset found!")
else:
    print("Dataset not found. Make sure to:")
    print("1. Add your dataset to the notebook (+ Add Data)")
    print("2. Update DATASET_PATH in configuration")
    print("\nAvailable datasets in /kaggle/input:")
    for item in os.listdir("/kaggle/input"):
        print(f"   - /kaggle/input/{item}")
    raise FileNotFoundError(f"Dataset path does not exist: {DATASET_PATH}")

# Robust loader: supports JSONL and pretty JSON arrays/objects.
try:
    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    print("Loaded with datasets JSON loader")
except Exception as e:
    print(f"Standard loader failed: {e}")
    print("Falling back to manual JSON parsing...")
    from datasets import Dataset

    # Read with utf-8-sig to transparently remove UTF-8 BOM if present.
    with open(DATASET_PATH, "rb") as f:
        raw = f.read().decode("utf-8-sig", errors="replace")

    # Extra safety in case BOM survives after decode.
    raw = raw.lstrip("\ufeff").strip()

    records = []
    if raw.startswith("[") or raw.startswith("{"):
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            records = parsed
        elif isinstance(parsed, dict):
            # If wrapped in a key like {'data': [...]}, use that list.
            list_value = next((v for v in parsed.values() if isinstance(v, list)), None)
            records = list_value if list_value is not None else [parsed]
    else:
        # JSONL fallback
        for line in raw.splitlines():
            line = line.strip().lstrip("\ufeff")
            if line:
                records.append(json.loads(line))

    if not records:
        raise ValueError("No records found in dataset file after fallback parsing.")

    # Normalize field types to avoid Arrow schema conflicts.
    normalized = []
    for r in records:
        inp = r.get("input", "")
        out = r.get("output", "")
        if isinstance(inp, (list, dict)):
            inp = json.dumps(inp, ensure_ascii=False)
        if isinstance(out, (list, dict)):
            out = json.dumps(out, ensure_ascii=False)
        normalized.append({"input": str(inp), "output": str(out)})

    dataset = Dataset.from_list(normalized)
    print("Loaded with fallback parser")

print(f"Total examples: {len(dataset):,}")

from collections import Counter
output_dist = Counter(dataset["output"])
print("\nOutput distribution:")
for cat, count in sorted(output_dist.items(), key=lambda x: -x[1]):
    print(f"   {cat}: {count}")

INSTRUCTION = (
    "Classify the instruction type into one of: SIMPLE INSTRUCTION, "
    "INSTRUCTION WITH SEQUENCE, PARALLEL INSTRUCTION, "
    "INSTRUCTION WITH PURPOSE, INSTRUCTION WITH REASON, "
    "EXCLUSIVE INSTRUCTION (OBJECTS), EXCLUSIVE INSTRUCTION (ACTIONS)."
)

def format_prompt(example):
    input_text = example.get("input", "")
    output = example.get("output", "")
    return {
        "text": (
            f"### Instruction:\n{INSTRUCTION}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n{output}"
        )
    }

dataset = dataset.map(format_prompt)
print("\nDataset formatted for Llama-3.1-8B!")

In [ ]:
# Loading Llama-3.1-8B with 4-bit quantization
print("Loading Llama-3.1-8B with 4-bit quantization...")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for Llama-3.1-8B fine-tuning in this notebook.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        token=HF_TOKEN,
        use_fast=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
    )

    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)
    print(f"Llama-3.1-8B loaded! Memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")
except Exception as e:
    raise RuntimeError(
        "Failed to load model/tokenizer. Check HF token access for gated model and GPU memory. "
        f"Original error: {e}"
    )

In [ ]:
# Applying LoRA adapters for Llama-3.1-8B
print("Applying LoRA adapters...")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    # Llama target modules
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print("LoRA applied to Llama-3.1-8B!")

In [ ]:
print("Tokenizing...")

def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")
print(f"Tokenized {len(tokenized_dataset)} examples")

In [ ]:
print("="*64)
print("STARTING LLAMA-3.1-8B TRAINING WITH SMART CHECKPOINTING")
print("="*64)

from torch.utils.data import DataLoader
from transformers import get_scheduler, DataCollatorForLanguageModeling
from tqdm.auto import tqdm
import glob
import time
import shutil

# Data pipeline
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataloader = DataLoader(
    tokenized_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=torch.cuda.is_available(),
)

CHECKPOINT_DIR = "/kaggle/working/checkpoints_llama31"
BEST_META_PATH = os.path.join(CHECKPOINT_DIR, "best_checkpoint.json")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _checkpoint_step(path):
    try:
        return int(path.rstrip("/").split("_")[-1])
    except Exception:
        return -1

def list_checkpoints():
    ckpts = glob.glob(f"{CHECKPOINT_DIR}/checkpoint_step_*")
    return sorted(ckpts, key=_checkpoint_step)

def latest_checkpoint():
    ckpts = list_checkpoints()
    return ckpts[-1] if ckpts else None

def prune_old_checkpoints(keep=2, protect_paths=None):
    protect_paths = set(p for p in (protect_paths or []) if p)
    ckpts = list_checkpoints()
    while len(ckpts) > keep:
        deletable = [p for p in ckpts if p not in protect_paths]
        if not deletable:
            break
        old_ckpt = deletable[0]
        ckpts.remove(old_ckpt)
        shutil.rmtree(old_ckpt, ignore_errors=True)
        print(f"Deleted old checkpoint: {old_ckpt}")

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
num_update_steps_per_epoch = max(1, len(train_dataloader) // GRAD_ACCUM_STEPS)
num_training_steps = num_update_steps_per_epoch * NUM_EPOCHS
lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

start_epoch = 0
global_step = 0
best_loss = float("inf")
best_step = -1
best_checkpoint_path = None

# Restore best-loss metadata if available
if os.path.exists(BEST_META_PATH):
    try:
        with open(BEST_META_PATH, "r", encoding="utf-8") as f:
            best_meta = json.load(f)
        best_loss = float(best_meta.get("best_loss", float("inf")))
        best_step = int(best_meta.get("best_step", -1))
        best_checkpoint_path = best_meta.get("best_checkpoint_path")
        print(f"Loaded best-loss metadata: loss={best_loss:.6f}, step={best_step}")
    except Exception as e:
        print(f"Could not read best checkpoint metadata: {e}")

best_effort_resume = latest_checkpoint()

# Smart resume: supports adapter-only checkpoints and training state
if best_effort_resume:
    print(f"Found checkpoint: {best_effort_resume}")
    try:
        training_state_path = os.path.join(best_effort_resume, "training_state.pt")
        adapter_path = os.path.join(best_effort_resume, "lora_adapter")

        if os.path.isdir(adapter_path):
            model.load_adapter(adapter_path, adapter_name="default", is_trainable=True)
            print("LoRA adapter restored")

        if os.path.exists(training_state_path):
            state = torch.load(training_state_path, map_location="cpu")
            if "optimizer" in state and "scheduler" in state:
                optimizer.load_state_dict(state["optimizer"])
                lr_scheduler.load_state_dict(state["scheduler"])
            start_epoch = int(state.get("epoch", 0))
            global_step = int(state.get("global_step", 0))
            state_loss = state.get("tracked_loss")
            if isinstance(state_loss, (float, int)) and state_loss < best_loss:
                best_loss = float(state_loss)
                best_step = global_step
                best_checkpoint_path = best_effort_resume
            print(f"Training state restored (epoch={start_epoch+1}, step={global_step})")
        else:
            print("training_state.pt missing, starting optimizer/scheduler fresh")
    except Exception as e:
        print(f"Could not fully resume checkpoint: {e}")
        print("Continuing with fresh optimizer/scheduler state.")
else:
    print("Starting fresh training (no checkpoint found)")

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

print("\nTraining Info:")
print(f"Total global steps: {num_training_steps} | Per epoch: {num_update_steps_per_epoch}")
print(f"Raw batches per epoch: {len(train_dataloader)}")
print(f"Batch size: {BATCH_SIZE} | Grad accumulation: {GRAD_ACCUM_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Save every {SAVE_EVERY_STEPS} global steps or {SAVE_EVERY_MINUTES} minutes")
print(f"Keep latest {KEEP_CHECKPOINTS} checkpoint(s)")
print(f"Checkpoints dir: {CHECKPOINT_DIR}")

def _write_best_meta():
    payload = {
        "best_loss": best_loss,
        "best_step": best_step,
        "best_checkpoint_path": best_checkpoint_path,
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with open(BEST_META_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def save_checkpoint(epoch, step, reason="periodic", include_optimizer_state=True, tracked_loss=None, mark_best=False):
    global best_checkpoint_path
    checkpoint_path = f"{CHECKPOINT_DIR}/checkpoint_step_{step}"
    os.makedirs(checkpoint_path, exist_ok=True)

    # Save LoRA adapter and tokenizer
    adapter_dir = os.path.join(checkpoint_path, "lora_adapter")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    state_payload = {
        "epoch": int(epoch),
        "global_step": int(step),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "reason": reason,
        "tracked_loss": tracked_loss if tracked_loss is not None else None,
    }

    if include_optimizer_state:
        state_payload["optimizer"] = optimizer.state_dict()
        state_payload["scheduler"] = lr_scheduler.state_dict()

    torch.save(state_payload, os.path.join(checkpoint_path, "training_state.pt"))
    print(f"\nCHECKPOINT SAVED [{reason}] step={step} -> {checkpoint_path}")

    if mark_best:
        best_checkpoint_path = checkpoint_path
        _write_best_meta()
        print(f"New best checkpoint: loss={best_loss:.6f} at step={best_step}")

    prune_old_checkpoints(keep=KEEP_CHECKPOINTS, protect_paths=[best_checkpoint_path])

def safe_save_on_error(epoch, step, error):
    try:
        print(f"\nTraining error captured at epoch={epoch+1}, step={step}: {error}")
        save_checkpoint(epoch, step, reason="error_recovery", include_optimizer_state=True, tracked_loss=best_loss)
        print("Recovery checkpoint saved successfully.")
    except Exception as save_err:
        print(f"Failed to save recovery checkpoint: {save_err}")

model.train()
last_save_time = time.time()

# Conservative skip logic when resuming
skip_batches = max(0, global_step * GRAD_ACCUM_STEPS)

try:
    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_loss = 0.0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=True)

        for step, batch in enumerate(progress_bar):
            if epoch == start_epoch and step < skip_batches:
                continue

            batch = {k: v.to(device) for k, v in batch.items()}

            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                outputs = model(**batch)
                loss = outputs.loss / GRAD_ACCUM_STEPS

            scaler.scale(loss).backward()
            epoch_loss += loss.item()

            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_dataloader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                lr_scheduler.step()
                global_step += 1

                avg_loss = epoch_loss / max(1, (step + 1))
                next_ckpt_step = ((global_step // SAVE_EVERY_STEPS) + 1) * SAVE_EVERY_STEPS
                progress_bar.set_postfix({
                    "loss": f"{avg_loss:.4f}",
                    "best": f"{best_loss:.4f}" if best_loss < float("inf") else "n/a",
                    "g_step": global_step,
                    "next_ckpt": next_ckpt_step,
                })

                # Best-loss tracking
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    best_step = global_step
                    save_checkpoint(
                        epoch,
                        global_step,
                        reason="best_loss",
                        tracked_loss=best_loss,
                        mark_best=True,
                    )
                    last_save_time = time.time()

                time_since_save = (time.time() - last_save_time) / 60.0
                should_save_by_step = (global_step % SAVE_EVERY_STEPS == 0)
                should_save_by_time = (time_since_save >= SAVE_EVERY_MINUTES)

                if should_save_by_step or should_save_by_time:
                    save_reason = "step" if should_save_by_step else "time"
                    save_checkpoint(
                        epoch,
                        global_step,
                        reason=save_reason,
                        tracked_loss=avg_loss,
                    )
                    last_save_time = time.time()

        epoch_avg_loss = epoch_loss / max(1, len(train_dataloader))
        print(f"Epoch {epoch+1} completed | Avg Loss: {epoch_avg_loss:.4f}")
        save_checkpoint(
            epoch,
            global_step,
            reason="epoch_end",
            tracked_loss=epoch_avg_loss,
        )
        last_save_time = time.time()
        skip_batches = 0

except KeyboardInterrupt as e:
    safe_save_on_error(epoch, global_step, f"KeyboardInterrupt: {e}")
    raise
except RuntimeError as e:
    # Handle CUDA OOM and transient runtime failures with a recovery checkpoint
    safe_save_on_error(epoch, global_step, f"RuntimeError: {e}")
    if "out of memory" in str(e).lower() and torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA cache cleared after OOM.")
    raise
except Exception as e:
    safe_save_on_error(epoch, global_step, f"UnexpectedError: {e}")
    raise
finally:
    # Always attempt one final save of current adapter snapshot
    try:
        save_checkpoint(
            max(0, epoch),
            global_step,
            reason="final_snapshot",
            tracked_loss=best_loss if best_loss < float("inf") else None,
        )
    except Exception as final_err:
        print(f"Final snapshot save failed: {final_err}")

print("\n" + "="*64)
print("LLAMA-3.1-8B TRAINING COMPLETE")
if best_loss < float("inf"):
    print(f"Best loss: {best_loss:.6f} at step {best_step}")
print("="*64)

In [ ]:
# Saving final model to Kaggle output
print("Saving final model to Kaggle...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save final LoRA adapter
adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved to: {adapter_path}")

# Also save in root /kaggle/working for easy access
backup_path = "/kaggle/working/llama31_lora_adapter"
model.save_pretrained(backup_path)
tokenizer.save_pretrained(backup_path)
print(f"Backup saved to: {backup_path}")

# Display saved files
print("\nSaved files:")
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"   {f}: {size:.2f} MB")

# List retained checkpoints (old ones already deleted by retention logic)
print("\nRetained checkpoints:")
for ckpt in sorted(glob.glob(f"{CHECKPOINT_DIR}/checkpoint_step_*")):
    step = ckpt.split("_")[-1]
    print(f"   checkpoint_step_{step}")

# Show best-loss checkpoint info if metadata exists
if os.path.exists(BEST_META_PATH):
    try:
        with open(BEST_META_PATH, "r", encoding="utf-8") as f:
            best_meta = json.load(f)
        print("\nBest checkpoint:")
        print(f"   Step: {best_meta.get('best_step', 'N/A')}")
        print(f"   Loss: {best_meta.get('best_loss', 'N/A')}")
        print(f"   Path: {best_meta.get('best_checkpoint_path', 'N/A')}")
    except Exception as e:
        print(f"Could not read best checkpoint metadata: {e}")

print("\nModel will be available in 'Output' section after notebook completes!")

In [ ]:
# Testing the fine-tuned Llama model
print("Testing Llama-3.1-8B...")

test_sentences = [
    ("Boil water", "SIMPLE INSTRUCTION"),
    ("Pour hot water then stir", "INSTRUCTION WITH SEQUENCE"),
    ("Add sugar and milk", "PARALLEL INSTRUCTION"),
    ("If you want strong coffee, add extra powder", "INSTRUCTION WITH PURPOSE"),
    ("Heat it because the water is cold", "INSTRUCTION WITH REASON"),
    ("Use instant coffee or filter coffee", "EXCLUSIVE INSTRUCTION (OBJECTS)"),
    ("Drink immediately or store in flask", "EXCLUSIVE INSTRUCTION (ACTIONS)"),
]

model.eval()
print("\n" + "="*70)
print("TEST RESULTS")
print("="*70)

correct = 0
for sentence, expected in test_sentences:
    prompt = f"### Instruction:\n{INSTRUCTION}\n\n### Input:\n{sentence}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = result.split("### Response:")[-1].strip().split("\n")[0].strip()

    match = "✓" if pred == expected else "✗"
    if pred == expected:
        correct += 1

    print(f"\nInput: {sentence}")
    print(f"Expected: {expected}")
    print(f"Predicted: {pred} {match}")

print("\n" + "="*70)
print(f"Testing complete! Accuracy: {correct}/{len(test_sentences)} ({100*correct/len(test_sentences):.1f}%)")
print("="*70)

In [ ]:
# Zipping the adapter for download
import shutil

zip_name = "llama31_lora_adapter"
zip_path = f"/kaggle/working/{zip_name}.zip"
shutil.make_archive(f"/kaggle/working/{zip_name}", "zip", adapter_path)

print(f"Model zipped: {zip_path}")
print(f"Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")

print("\n" + "="*60)
print("HOW TO DOWNLOAD YOUR MODEL:")
print("="*60)
print("1. Go to 'Output' section (right side panel)")
print("2. Click on 'llama31_lora_adapter.zip'")
print("3. Download and extract")
print("\nThe model will also be saved automatically when")
print("you click 'Save Version' with 'Save Output' enabled!")
print("="*60)